# KArSL Keypoints and Pose Manifest Verification

Read-only verification of packaged keypoints subset and external Pose manifests. Raw NPY/Pose files are not in this package and cannot be re-audited here.


In [ ]:
from pathlib import Path
import os
import pandas as pd
root = Path(os.environ.get("SIGNOVA_KARSL_ROOT", ".")).expanduser().resolve()
if not (root / "data/metadata/karsl_metadata.csv").is_file():
    candidate = Path.cwd() / "karsl"
    if (candidate / "data/metadata/karsl_metadata.csv").is_file():
        root = candidate.resolve()
    else:
        raise FileNotFoundError("Set SIGNOVA_KARSL_ROOT to the unpacked karsl folder")
print("KArSL root:", root)


In [ ]:
k = pd.read_csv(root / "data/metadata/keypoints_manifest.csv", low_memory=False)
p = pd.read_csv(root / "data/pose/karsl_pose_manifest.csv", low_memory=False)
s = pd.read_csv(root / "data/pose/karsl_pose_signer_folds.csv", low_memory=False)
quality = pd.read_csv(root / "data/reports/karsl_pose_502_class_quality_audit.csv", low_memory=False)
checks = {
    "Keypoints manifest nonempty": len(k) > 0,
    "Pose manifest records": len(p) == 75515,
    "Pose fold records": len(s) == 226545,
    "Three Pose folds": s["fold"].nunique() == 3,
    "502 Pose classes recorded": len(quality) == 502,
    "502 sampled Pose file audits present": quality.shape[0] == 502
}
for name, ok in checks.items():
    print(name, "PASS" if ok else "FAIL")
assert all(checks.values())
print("Pose binary content revalidated in this notebook: NO")
